# Delaunay quickstart

From the repository root, set up the notebook environment and launch this notebook with:

```bash
just notebook-setup
just notebook
```

This notebook generates a seeded 1000-vertex 3D Euclidean Delaunay triangulation from ball-distributed points with the Rust `delaunay` CLI, loads the generic simplicial-complex JSON export, summarizes the mesh, and renders a first 3D wireframe view. It also runs the companion convex-hull export and writes a transparent PNG preview under `target/notebooks/00_quickstart/`.

Reviewer-facing release instructions and paper-claim mapping that consume this visual-inspection workflow are tracked in [#408](https://github.com/acgetchell/delaunay/issues/408).

The default is intentionally CI-friendly. For a cube-distributed point set, set `DELAUNAY_QUICKSTART_DISTRIBUTION=cube`. For a larger interactive stress view, start Jupyter with `DELAUNAY_QUICKSTART_VERTICES=9000` or another positive value.

## 1. Generate a seeded 3D triangulation

The helper below honors `DELAUNAY_BINARY` when you want to point at a prebuilt binary. Otherwise it prefers `target/perf/delaunay` and falls back to `cargo run --profile perf --features cli --bin delaunay -- ...`.

In [ ]:
"""Generate a seeded 3D Delaunay triangulation artifact."""

import json
import math
import os
import shutil
import subprocess
import time
from itertools import combinations
from pathlib import Path
from typing import Any, Never, cast

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection


def find_repo_root(start: Path) -> Path:
    """Return the repository root containing the project marker files."""
    for path in (start, *start.parents):
        if (path / "Cargo.toml").is_file() and (path / "pyproject.toml").is_file():
            return path
    message = "Run this notebook from inside the delaunay repository."
    raise RuntimeError(message)


def positive_int_env(name: str, default: int) -> int:
    """Read a positive integer environment override."""
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be a positive integer, got {raw_value!r}") from error
    if value <= 0:
        raise ValueError(f"{name} must be positive, got {value}")
    return value


def uint64_env(name: str, default: int) -> int:
    """Read an unsigned 64-bit integer environment override."""
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default
    try:
        value = int(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be an unsigned 64-bit integer, got {raw_value!r}") from error
    if not 0 <= value <= 2**64 - 1:
        raise ValueError(f"{name} must be in [0, 2^64 - 1], got {value}")
    return value


def nonnegative_float_env(name: str, default: float) -> float:
    """Read a finite non-negative float environment override."""
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default
    try:
        value = float(raw_value)
    except ValueError as error:
        raise ValueError(f"{name} must be a finite non-negative float, got {raw_value!r}") from error
    if not math.isfinite(value) or value < 0.0:
        raise ValueError(f"{name} must be finite and non-negative, got {value}")
    return value


def string_choice_env(name: str, default: str, choices: set[str]) -> str:
    """Read a lower-case string choice environment override."""
    value = os.environ.get(name, default).strip().lower()
    if value not in choices:
        message = f"{name} must be one of {sorted(choices)}, got {value!r}"
        raise ValueError(message)
    return value


def bool_env(name: str, *, default: bool) -> bool:
    """Read an explicit boolean environment override."""
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default
    value = raw_value.strip().lower()
    if value in {"1", "true", "yes", "on"}:
        return True
    if value in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be a boolean value, got {raw_value!r}")


def repo_output_path_env(name: str, root: Path, default: Path) -> Path:
    """Read a repository-relative output path override."""
    raw_value = os.environ.get(name)
    if raw_value is None:
        return default
    if not raw_value.strip():
        raise ValueError(f"{name} must not be empty")
    path = Path(raw_value).expanduser()
    if path.is_absolute():
        return path
    return root / path


def run_command(command: list[str], *, cwd: Path, timeout: int) -> subprocess.CompletedProcess[str]:
    """Run one repository command with captured output and a timeout."""
    print("$", " ".join(command))
    result = subprocess.run(  # noqa: S603 - command is an argv list with shell=False and controlled cwd.
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        message = f"command failed with exit code {result.returncode}: {' '.join(command)}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        raise RuntimeError(message)
    return result


def delaunay_command_prefix(root: Path) -> list[str]:
    """Return the command prefix for the local `delaunay` CLI."""
    configured = os.environ.get("DELAUNAY_BINARY")
    if configured is not None:
        if not configured.strip():
            message = "DELAUNAY_BINARY must not be empty"
            raise ValueError(message)
        binary = Path(configured).expanduser()
        binary = binary if binary.is_absolute() else root / binary
        if not binary.is_file():
            raise FileNotFoundError(f"DELAUNAY_BINARY does not point to a file: {binary}")
        return [str(binary)]

    binary_name = "delaunay.exe" if os.name == "nt" else "delaunay"
    local_binary = root / "target" / "perf" / binary_name
    if local_binary.is_file():
        return [str(local_binary)]

    cargo = shutil.which("cargo")
    if cargo is None:
        message = "cargo executable was not found on PATH; set DELAUNAY_BINARY to a built binary"
        raise RuntimeError(message)
    return [cargo, "run", "--profile", "perf", "--features", "cli", "--bin", "delaunay", "--"]


ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOK_STEM = "00_quickstart"
OUTPUT_DIR = ROOT / "target" / "notebooks" / NOTEBOOK_STEM
VISUALIZATION_PATH = OUTPUT_DIR / "visualization_3d.json"
FIGURE_PATH = OUTPUT_DIR / "triangulation_3d_wireframe.png"
VERTEX_COUNT = positive_int_env("DELAUNAY_QUICKSTART_VERTICES", 1000)
SEED = uint64_env("DELAUNAY_QUICKSTART_SEED", 873)
DISTRIBUTION = string_choice_env("DELAUNAY_QUICKSTART_DISTRIBUTION", "ball", {"ball", "cube"})
EDGE_LIMIT = positive_int_env("DELAUNAY_QUICKSTART_EDGE_LIMIT", 25_000)
POINT_LIMIT = positive_int_env("DELAUNAY_QUICKSTART_POINT_LIMIT", 10_000)
TIMEOUT_SECONDS = positive_int_env("DELAUNAY_QUICKSTART_TIMEOUT_SECONDS", 600)
EDGE_ALPHA = nonnegative_float_env("DELAUNAY_QUICKSTART_EDGE_ALPHA", 0.045)
POINT_ALPHA = nonnegative_float_env("DELAUNAY_QUICKSTART_POINT_ALPHA", 0.28)
POINT_SIZE = nonnegative_float_env("DELAUNAY_QUICKSTART_POINT_SIZE", 2.2)
HULL_ALPHA = nonnegative_float_env("DELAUNAY_QUICKSTART_HULL_ALPHA", 0.0)
TRANSPARENT_BACKGROUND = bool_env("DELAUNAY_QUICKSTART_TRANSPARENT", default=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    *delaunay_command_prefix(ROOT),
    "generate",
    "visualization",
    "--dimension",
    "3",
    "--vertices",
    str(VERTEX_COUNT),
    "--distribution",
    DISTRIBUTION,
    "--seed",
    str(SEED),
    "--output",
    str(VISUALIZATION_PATH),
]

start_time = time.perf_counter()
run_command(command, cwd=ROOT, timeout=TIMEOUT_SECONDS)
elapsed = time.perf_counter() - start_time

print(f"Repository: {ROOT}")
print(f"Visualization JSON: {VISUALIZATION_PATH}")
print(f"Generated {VERTEX_COUNT} 3D {DISTRIBUTION} vertices with seed {SEED} in {elapsed:.2f}s")

## 2. Load and summarize the triangulation

The CLI writes the generic `delaunay.simplicial_complex` visualization schema rather than the invariant-bearing TDS hydration snapshot. The parser below consumes only stable vertex IDs, coordinates, simplex membership, and facet adjacency, so downstream producers can reuse the plotting path.


In [ ]:
"""Load the generated generic visualization schema and derive display geometry."""

type JsonObject = dict[str, Any]
type Point3 = tuple[float, float, float]
type SimplexVertices = tuple[str, str, str, str]
type NeighborSlots = tuple[str | None, str | None, str | None, str | None]
type Edge = tuple[str, str]
type Face = tuple[str, str, str]
type AxisLimits = tuple[tuple[float, float], tuple[float, float], tuple[float, float]]


def reject_json_constant(value: str) -> object:
    """Reject non-standard JSON numeric constants before artifact parsing."""
    raise ValueError(f"JSON artifact contains non-finite value {value}")


def load_json_object(path: Path) -> JsonObject:
    """Load a JSON object from disk."""
    with path.open(encoding="utf-8") as file:
        value = json.load(file, parse_constant=reject_json_constant)
    if not isinstance(value, dict):
        message = f"expected top-level JSON object in {path}"
        raise TypeError(message)
    return cast("JsonObject", value)


def finite_coordinate(value: object, context: str) -> float:
    """Parse one finite coordinate from a Rust JSON export."""
    if isinstance(value, bool) or not isinstance(value, int | float):
        message = f"{context} must be a finite JSON number, got {value!r}"
        raise TypeError(message)
    coordinate = float(value)
    if not math.isfinite(coordinate):
        message = f"{context} must be finite, got {coordinate!r}"
        raise ValueError(message)
    return coordinate


def point3(value: object) -> Point3:
    """Parse one finite 3D coordinate array."""
    if not isinstance(value, list) or len(value) != 3:
        message = f"expected 3D point coordinates, got {value!r}"
        raise TypeError(message)
    return (
        finite_coordinate(value[0], "point.x"),
        finite_coordinate(value[1], "point.y"),
        finite_coordinate(value[2], "point.z"),
    )


def fail_type(message: str) -> Never:
    """Raise a typed schema error without hiding the offending message."""
    raise TypeError(message)


def visualization_metadata(data: JsonObject) -> JsonObject:
    """Validate and return v1 generic visualization metadata."""
    metadata = data.get("metadata")
    if not isinstance(metadata, dict):
        fail_type("visualization field 'metadata' must be an object")
    if metadata.get("schema") != "delaunay.simplicial_complex":
        raise ValueError(f"unsupported visualization schema: {metadata.get('schema')!r}")
    if metadata.get("schema_version") != 1:
        raise ValueError(f"unsupported visualization schema version: {metadata.get('schema_version')!r}")
    if metadata.get("dimension") != 3:
        raise ValueError(f"expected 3D visualization data, got {metadata.get('dimension')!r}")
    return cast("JsonObject", metadata)


def vertex_coordinates(data: JsonObject) -> dict[str, Point3]:
    """Return a stable-ID-to-coordinate map from generic visualization data."""
    records = data.get("vertices")
    if not isinstance(records, list):
        fail_type("visualization field 'vertices' must be a list")

    coordinates: dict[str, Point3] = {}
    for record in records:
        if not isinstance(record, dict):
            fail_type("vertex record must be a JSON object")
        vertex_id = record.get("id")
        if not isinstance(vertex_id, str):
            fail_type("vertex record is missing string field 'id'")
        if vertex_id in coordinates:
            raise ValueError(f"duplicate vertex ID: {vertex_id}")
        coordinates[vertex_id] = point3(record.get("coordinates"))
    return coordinates


def simplex_vertex_map(data: JsonObject) -> dict[str, SimplexVertices]:
    """Return simplex IDs mapped to ordered vertex membership."""
    records = data.get("simplices")
    if not isinstance(records, list):
        fail_type("visualization field 'simplices' must be a list")

    simplices: dict[str, SimplexVertices] = {}
    for record in records:
        if not isinstance(record, dict):
            fail_type("simplex record must be a JSON object")
        simplex_id = record.get("id")
        vertex_ids = record.get("vertex_ids")
        if not isinstance(simplex_id, str):
            fail_type("simplex record is missing string field 'id'")
        if not isinstance(vertex_ids, list) or len(vertex_ids) != 4 or not all(isinstance(item, str) for item in vertex_ids):
            fail_type("3D simplex records must contain four string vertex IDs")
        if simplex_id in simplices:
            raise ValueError(f"duplicate simplex ID: {simplex_id}")
        typed_ids = cast("list[str]", vertex_ids)
        simplices[simplex_id] = (typed_ids[0], typed_ids[1], typed_ids[2], typed_ids[3])
    return simplices


def simplex_neighbor_map(data: JsonObject, simplices: dict[str, SimplexVertices]) -> dict[str, NeighborSlots]:
    """Build opposite-facet neighbor slots from generic adjacency records."""
    records = data.get("adjacency")
    if not isinstance(records, list):
        fail_type("visualization field 'adjacency' must be a list")

    slots: dict[str, list[str | None]] = {simplex_id: [None, None, None, None] for simplex_id in simplices}
    seen: set[tuple[str, int]] = set()
    for record in records:
        if not isinstance(record, dict):
            fail_type("adjacency record must be a JSON object")
        simplex_id = record.get("simplex_id")
        facet_index = record.get("facet_index")
        neighbor_id = record.get("neighbor_simplex_id")
        if not isinstance(simplex_id, str) or simplex_id not in slots:
            raise KeyError(f"adjacency references unknown simplex ID: {simplex_id!r}")
        if isinstance(facet_index, bool) or not isinstance(facet_index, int) or not 0 <= facet_index < 4:
            raise ValueError(f"invalid 3D facet index: {facet_index!r}")
        if neighbor_id is not None and not isinstance(neighbor_id, str):
            fail_type("neighbor simplex ID must be a string or null")
        key = (simplex_id, facet_index)
        if key in seen:
            raise ValueError(f"duplicate adjacency record: {key}")
        seen.add(key)
        slots[simplex_id][facet_index] = neighbor_id
    expected = len(simplices) * 4
    if len(seen) != expected:
        raise ValueError(f"expected {expected} adjacency records, got {len(seen)}")
    return {simplex_id: (values[0], values[1], values[2], values[3]) for simplex_id, values in slots.items()}


def validate_visualization_references(
    coordinates: dict[str, Point3],
    simplices: dict[str, SimplexVertices],
    neighbors: dict[str, NeighborSlots],
) -> None:
    """Ensure generic visualization records agree before plotting."""
    missing_vertices = sorted({vertex_id for vertex_ids in simplices.values() for vertex_id in vertex_ids if vertex_id not in coordinates})
    if missing_vertices:
        raise KeyError(f"simplices reference {len(missing_vertices)} unknown vertex ID(s), first: {missing_vertices[:5]}")
    unknown_neighbors = sorted(
        {neighbor_id for values in neighbors.values() for neighbor_id in values if neighbor_id is not None and neighbor_id not in simplices}
    )
    if unknown_neighbors:
        raise KeyError(f"adjacency references {len(unknown_neighbors)} unknown simplex ID(s), first: {unknown_neighbors[:5]}")


def unique_edges(simplices: dict[str, SimplexVertices]) -> list[Edge]:
    """Derive sorted unique graph edges from tetrahedral simplex records."""
    edges: set[Edge] = set()
    for vertex_ids in simplices.values():
        for left, right in combinations(vertex_ids, 2):
            edges.add(cast("Edge", tuple(sorted((left, right)))))
    return sorted(edges)


def coordinate_axis_limits(points: list[Point3]) -> AxisLimits:
    """Return equal-scale 3D plot limits for a point cloud."""
    if not points:
        message = "cannot compute display limits for an empty point cloud"
        raise ValueError(message)
    minima = [min(point[axis] for point in points) for axis in range(3)]
    maxima = [max(point[axis] for point in points) for axis in range(3)]
    span = max(maximum - minimum for minimum, maximum in zip(minima, maxima, strict=True))
    half_span = max(span * 1.18, 0.5)
    centers = [(minimum + maximum) / 2.0 for minimum, maximum in zip(minima, maxima, strict=True)]
    return (
        (centers[0] - half_span, centers[0] + half_span),
        (centers[1] - half_span, centers[1] + half_span),
        (centers[2] - half_span, centers[2] + half_span),
    )


def scaled_axis_limits(limits: AxisLimits, scale: float) -> AxisLimits:
    """Return equal-scale limits scaled around their shared center."""
    if not math.isfinite(scale) or scale <= 0.0:
        raise ValueError(f"axis limit scale must be finite and positive, got {scale}")
    expanded_limits = []
    for lower, upper in limits:
        center = (lower + upper) / 2.0
        half_span = (upper - lower) * scale / 2.0
        expanded_limits.append((center - half_span, center + half_span))
    return (expanded_limits[0], expanded_limits[1], expanded_limits[2])


def normalized_point(point: Point3, limits: AxisLimits) -> Point3:
    """Map one point into display-normalized coordinates."""
    coords = []
    for coordinate, (lower, upper) in zip(point, limits, strict=True):
        width = upper - lower
        coords.append(0.5 if width == 0.0 else (coordinate - lower) / width)
    return (coords[0], coords[1], coords[2])


def apply_axis_limits(axis: Any, limits: AxisLimits) -> None:
    """Apply equal-scale 3D plot limits to one Matplotlib axis."""
    axis.set_xlim(*limits[0])
    axis.set_ylim(*limits[1])
    axis.set_zlim(*limits[2])
    axis.set_box_aspect((1.0, 1.0, 1.0))


def edge_sort_key(edge: Edge, point_map: dict[str, Point3]) -> tuple[float, float, float, float]:
    """Return a coordinate-based display key for one edge."""
    left, right = edge
    left_point = point_map[left]
    right_point = point_map[right]
    midpoint = tuple((left_point[index] + right_point[index]) / 2.0 for index in range(3))
    length_squared = sum((left_point[index] - right_point[index]) ** 2 for index in range(3))
    return (midpoint[0], midpoint[1], midpoint[2], length_squared)


def boundary_faces(simplices: dict[str, SimplexVertices], neighbors: dict[str, NeighborSlots]) -> list[Face]:
    """Derive boundary triangles from null opposite-facet neighbor slots."""
    faces: set[Face] = set()
    for simplex_id, vertex_ids in simplices.items():
        slots = neighbors.get(simplex_id)
        if slots is None:
            raise KeyError(f"missing neighbor slots for simplex {simplex_id!r}")
        for opposite_index, neighbor_id in enumerate(slots):
            if neighbor_id is None:
                face_ids = tuple(vertex_id for index, vertex_id in enumerate(vertex_ids) if index != opposite_index)
                if len(face_ids) == 3:
                    faces.add(cast("Face", tuple(sorted(face_ids))))
    return sorted(faces)


visualization = load_json_object(VISUALIZATION_PATH)
metadata = visualization_metadata(visualization)
coordinates = vertex_coordinates(visualization)
simplices = simplex_vertex_map(visualization)
neighbors = simplex_neighbor_map(visualization, simplices)
validate_visualization_references(coordinates, simplices, neighbors)
edges = unique_edges(simplices)
faces = boundary_faces(simplices, neighbors)
AXIS_LIMITS = coordinate_axis_limits(list(coordinates.values()))

if len(coordinates) != VERTEX_COUNT:
    message = f"expected {VERTEX_COUNT} vertices, got {len(coordinates)}"
    raise RuntimeError(message)

print(f"vertices:          {len(coordinates):,}")
print(f"tetrahedra:        {len(simplices):,}")
print(f"unique edges:      {len(edges):,}")
print(f"boundary triangles:{len(faces):,}")
print(f"topology:          {metadata.get('topology_kind')} / {metadata.get('topology_guarantee')}")
print(f"JSON size:         {VISUALIZATION_PATH.stat().st_size / 1_048_576:.2f} MiB")
print(f"plot x-limits:     [{AXIS_LIMITS[0][0]:.3f}, {AXIS_LIMITS[0][1]:.3f}]")

## 3. Display the triangulation

The exploratory plot emphasizes internal structure: it renders the Delaunay edge graph as a transparent wireframe and keeps hull faces off by default. The source triangulation is still the full generated 3D Delaunay triangulation. For a surface hint, set `DELAUNAY_QUICKSTART_HULL_ALPHA=0.04`; for a denser or lighter exploratory view, tune `DELAUNAY_QUICKSTART_VERTICES`, `DELAUNAY_QUICKSTART_EDGE_ALPHA`, and `DELAUNAY_QUICKSTART_POINT_ALPHA` before launching Jupyter.

In [ ]:
"""Render a compact 3D view of the generated triangulation."""

display_vertex_ids = sorted(coordinates, key=coordinates.__getitem__)[:POINT_LIMIT]
display_edges = sorted(edges, key=lambda edge: edge_sort_key(edge, coordinates))[:EDGE_LIMIT]
point_cloud = [coordinates[vertex_id] for vertex_id in display_vertex_ids]
edge_segments = [[coordinates[left], coordinates[right]] for left, right in display_edges]
face_polygons = [[coordinates[vertex_id] for vertex_id in face] for face in faces]

xs = [point[0] for point in point_cloud]
ys = [point[1] for point in point_cloud]
zs = [point[2] for point in point_cloud]

figure = plt.figure(figsize=(8.0, 7.0), layout="constrained")
if TRANSPARENT_BACKGROUND:
    figure.patch.set_alpha(0.0)
axis = figure.add_subplot(111, projection="3d")
axis.scatter(
    xs,
    ys,
    zs,
    s=POINT_SIZE,
    alpha=POINT_ALPHA,
    color="#1f77b4",
    depthshade=False,
)

if edge_segments:
    edge_collection = Line3DCollection(
        edge_segments,
        colors="#3366aa",
        linewidths=0.26,
        alpha=EDGE_ALPHA,
    )
    axis.add_collection3d(edge_collection)
if face_polygons and HULL_ALPHA > 0.0:
    face_collection = Poly3DCollection(
        face_polygons,
        facecolors="#f58518",
        edgecolors="#f58518",
        linewidths=0.18,
        alpha=HULL_ALPHA,
    )
    axis.add_collection3d(face_collection)

axis.set_axis_off()
apply_axis_limits(axis, AXIS_LIMITS)
axis.set_proj_type("ortho")
axis.view_init(elev=22.0, azim=35.0)

figure.savefig(
    FIGURE_PATH,
    dpi=220,
    transparent=TRANSPARENT_BACKGROUND,
    bbox_inches="tight",
    pad_inches=0.02,
)
plt.show()

print(f"displayed vertices: {len(point_cloud):,} / {len(coordinates):,}")
print(f"displayed edges:    {len(edge_segments):,} / {len(edges):,}")
print(f"boundary faces:     {len(face_polygons):,} (alpha={HULL_ALPHA})")
print(f"saved figure:       {FIGURE_PATH}")

## 4. Generate the README figure

The README artifact uses the same seeded 3D point set. It renders the convex hull as randomly colored boundary facets and the Delaunay triangulation as a transparent cutaway, then saves the combined transparent PNG image to `target/notebooks/00_quickstart/delaunay_3d_readme.png` by default.

In [ ]:
"""Generate the README image from 3D Delaunay and convex-hull exports."""

type HullFacet = list[Point3]


def convex_hull_facets(data: JsonObject) -> list[HullFacet]:
    """Parse 3D convex-hull facet polygons from the CLI export."""
    if data.get("dimension") != 3:
        message = f"expected 3D convex hull export, got dimension {data.get('dimension')!r}"
        raise TypeError(message)
    raw_facets = data.get("facets")
    if not isinstance(raw_facets, list):
        message = "convex hull export field 'facets' must be a list"
        raise TypeError(message)

    hull_facets: list[HullFacet] = []
    for record in raw_facets:
        if not isinstance(record, dict):
            message = "convex hull facet record must be a JSON object"
            raise TypeError(message)
        raw_coordinates = record.get("coordinates")
        if not isinstance(raw_coordinates, list) or len(raw_coordinates) != 3:
            message = "3D convex hull facets must contain three coordinate arrays"
            raise TypeError(message)
        hull_facets.append([point3(value) for value in raw_coordinates])
    return hull_facets


def face_centroid(face: Face, point_map: dict[str, Point3]) -> Point3:
    """Return the centroid of one triangular face."""
    points = [point_map[vertex_id] for vertex_id in face]
    return (
        sum(point[0] for point in points) / 3.0,
        sum(point[1] for point in points) / 3.0,
        sum(point[2] for point in points) / 3.0,
    )


def polygon_centroid(points: HullFacet) -> Point3:
    """Return the centroid of one triangle polygon."""
    return (
        sum(point[0] for point in points) / 3.0,
        sum(point[1] for point in points) / 3.0,
        sum(point[2] for point in points) / 3.0,
    )


def internal_faces(simplices: dict[str, SimplexVertices], neighbors: dict[str, NeighborSlots]) -> list[Face]:
    """Derive unique interior triangles from non-null opposite-facet neighbor slots."""
    face_set: set[Face] = set()
    for simplex_id, vertex_ids in simplices.items():
        slots = neighbors.get(simplex_id)
        if slots is None:
            raise KeyError(f"missing neighbor slots for simplex {simplex_id!r}")
        for opposite_index, neighbor_id in enumerate(slots):
            if neighbor_id is None:
                continue
            face_ids = tuple(vertex_id for index, vertex_id in enumerate(vertex_ids) if index != opposite_index)
            if len(face_ids) == 3:
                face_set.add(cast("Face", tuple(sorted(face_ids))))
    return sorted(face_set)


def display_priority(point: Point3) -> tuple[float, float, float, float, float]:
    """Return a deterministic pseudo-random display priority from coordinates."""
    unit_point = normalized_point(point, AXIS_LIMITS)
    mixed = (unit_point[0] * 13.0 + unit_point[1] * 17.0 + unit_point[2] * 19.0) % 1.0
    centered_distance = sum((coordinate - 0.5) ** 2 for coordinate in unit_point)
    return (abs(mixed - 0.5), centered_distance, unit_point[0], unit_point[1], unit_point[2])


def face_display_priority(face: Face, point_map: dict[str, Point3]) -> tuple[float, float, float, float, float]:
    """Return a deterministic display priority for one face."""
    return display_priority(face_centroid(face, point_map))


def edge_display_priority(edge: Edge, point_map: dict[str, Point3]) -> tuple[float, float, float, float, float]:
    """Return a deterministic display priority for one edge."""
    left, right = edge
    left_point = point_map[left]
    right_point = point_map[right]
    midpoint = cast(
        "Point3",
        tuple((left_point[index] + right_point[index]) / 2.0 for index in range(3)),
    )
    return display_priority(midpoint)


def in_readme_cutaway(point: Point3) -> bool:
    """Return whether a centroid belongs to the README cutaway volume."""
    unit_point = normalized_point(point, AXIS_LIMITS)
    return 0.28 <= unit_point[0] <= 0.70 and 0.14 <= unit_point[1] <= 0.90 and 0.10 <= unit_point[2] <= 0.94


def colormap_samples(name: str, count: int) -> list[tuple[float, ...]]:
    """Return discrete samples from one Matplotlib colormap."""
    colormap = plt.get_cmap(name)
    return [cast("tuple[float, ...]", colormap(index)) for index in range(count)]


def palette_colors(count: int, *, stride: int = 7, offset: int = 3) -> list[tuple[float, ...]]:
    """Return a deterministic bright color cycle for facets."""
    palette = colormap_samples("tab20", 20) + colormap_samples("Set3", 12) + colormap_samples("Paired", 12)
    return [palette[(index * stride + offset) % len(palette)] for index in range(count)]


CONVEX_HULL_PATH = OUTPUT_DIR / "convex_hull_3d.json"
README_FIGURE_PATH = repo_output_path_env(
    "DELAUNAY_README_FIGURE",
    ROOT,
    OUTPUT_DIR / "delaunay_3d_readme.png",
)
README_INTERNAL_FACE_LIMIT = positive_int_env("DELAUNAY_README_INTERNAL_FACE_LIMIT", 700)
README_EDGE_LIMIT = positive_int_env("DELAUNAY_README_EDGE_LIMIT", 2_800)
README_POINT_LIMIT = positive_int_env("DELAUNAY_README_POINT_LIMIT", 700)
README_AXIS_SCALE = nonnegative_float_env("DELAUNAY_README_AXIS_SCALE", 0.30)
if README_AXIS_SCALE <= 0.0:
    raise ValueError(f"DELAUNAY_README_AXIS_SCALE must be positive, got {README_AXIS_SCALE}")
README_AXIS_LIMITS = scaled_axis_limits(AXIS_LIMITS, README_AXIS_SCALE)
README_TRANSPARENT = bool_env("DELAUNAY_README_TRANSPARENT", default=True)
README_BACKGROUND = "none" if README_TRANSPARENT else "black"

hull_command = [
    *delaunay_command_prefix(ROOT),
    "generate",
    "convex-hull",
    "--dimension",
    "3",
    "--vertices",
    str(VERTEX_COUNT),
    "--distribution",
    DISTRIBUTION,
    "--seed",
    str(SEED),
    "--output",
    str(CONVEX_HULL_PATH),
]

hull_start_time = time.perf_counter()
run_command(hull_command, cwd=ROOT, timeout=TIMEOUT_SECONDS)
hull_elapsed = time.perf_counter() - hull_start_time

hull_export = load_json_object(CONVEX_HULL_PATH)
hull_polygons = sorted(convex_hull_facets(hull_export), key=polygon_centroid)
all_internal_faces = internal_faces(simplices, neighbors)
cutaway_faces = [face for face in all_internal_faces if in_readme_cutaway(face_centroid(face, coordinates))]
cutaway_faces = sorted(cutaway_faces, key=lambda face: face_display_priority(face, coordinates))[:README_INTERNAL_FACE_LIMIT]
cutaway_edges = [
    edge
    for edge in edges
    if in_readme_cutaway(
        cast(
            "Point3",
            tuple((coordinates[edge[0]][index] + coordinates[edge[1]][index]) / 2.0 for index in range(3)),
        )
    )
]
cutaway_edges = sorted(cutaway_edges, key=lambda edge: edge_display_priority(edge, coordinates))[:README_EDGE_LIMIT]
readme_points = sorted(coordinates.values(), key=display_priority)[:README_POINT_LIMIT]

readme_figure = plt.figure(figsize=(12.0, 6.0), facecolor=README_BACKGROUND)
axes = [
    readme_figure.add_subplot(1, 2, 1, projection="3d"),
    readme_figure.add_subplot(1, 2, 2, projection="3d"),
]
readme_figure.subplots_adjust(left=0.055, right=0.945, bottom=0.11, top=0.89, wspace=0.08)
triangulation_axis, hull_axis = axes

for readme_axis in axes:
    readme_axis.set_facecolor(README_BACKGROUND)
    readme_axis.set_axis_off()
    apply_axis_limits(readme_axis, README_AXIS_LIMITS)
    readme_axis.set_proj_type("ortho")
    readme_axis.view_init(elev=20.0, azim=37.0)

hull_axis.add_collection3d(
    Poly3DCollection(
        hull_polygons,
        facecolors=palette_colors(len(hull_polygons), stride=5, offset=1),
        edgecolors="#050505",
        linewidths=0.36,
        alpha=0.9,
        zsort="average",
    )
)
hull_vertices = sorted({point for polygon in hull_polygons for point in polygon}, key=display_priority)
hull_axis.scatter(
    [point[0] for point in hull_vertices],
    [point[1] for point in hull_vertices],
    [point[2] for point in hull_vertices],
    s=6.0,
    color="#ef4444",
    alpha=0.96,
    depthshade=False,
)

boundary_polygons = [[coordinates[vertex_id] for vertex_id in face] for face in faces]
if boundary_polygons:
    triangulation_axis.add_collection3d(
        Poly3DCollection(
            boundary_polygons,
            facecolors="#14b8a6",
            edgecolors="#22d3ee",
            linewidths=0.14,
            alpha=0.045,
            zsort="average",
        )
    )
cutaway_polygons = [[coordinates[vertex_id] for vertex_id in face] for face in cutaway_faces]
if cutaway_polygons:
    triangulation_axis.add_collection3d(
        Poly3DCollection(
            cutaway_polygons,
            facecolors=palette_colors(len(cutaway_polygons), stride=7, offset=3),
            edgecolors="#060606",
            linewidths=0.10,
            alpha=0.18,
            zsort="average",
        )
    )
triangulation_axis.add_collection3d(
    Line3DCollection(
        [[coordinates[left], coordinates[right]] for left, right in cutaway_edges],
        colors="#60a5fa",
        linewidths=0.20,
        alpha=0.21,
    )
)
triangulation_axis.scatter(
    [point[0] for point in readme_points],
    [point[1] for point in readme_points],
    [point[2] for point in readme_points],
    s=1.8,
    color="#f43f5e",
    alpha=0.32,
    depthshade=False,
)

README_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
readme_figure.savefig(
    README_FIGURE_PATH,
    dpi=200,
    facecolor=readme_figure.get_facecolor(),
    transparent=README_TRANSPARENT,
)
plt.show()

print(f"convex hull facets:   {len(hull_polygons):,}")
print(f"boundary triangles:   {len(faces):,}")
print(f"internal faces drawn: {len(cutaway_polygons):,} / {len(all_internal_faces):,}")
print(f"cutaway edges drawn:  {len(cutaway_edges):,} / {len(edges):,}")
print(f"convex hull export:   {CONVEX_HULL_PATH}")
print(f"convex hull time:     {hull_elapsed:.2f}s")
print(f"README figure:        {README_FIGURE_PATH}")

## 5. Inspect artifacts or scale up

The generated JSON, exploratory wireframe, convex-hull export, and transparent README hero preview stay under `target/notebooks/00_quickstart/` during normal notebook runs. The source notebook remains output-free, and `just clean` removes these target artifacts.

To tune the transparent cutaway preview before launching Jupyter:

```bash
DELAUNAY_README_AXIS_SCALE=0.30 DELAUNAY_README_INTERNAL_FACE_LIMIT=500 DELAUNAY_README_EDGE_LIMIT=1800 just notebook
```

For a larger local 3D smoke test, relaunch the notebook with:

```bash
DELAUNAY_QUICKSTART_VERTICES=9000 just notebook
```

The larger run uses the same artifact paths under `target/notebooks/00_quickstart/`, so it can replace the default preview artifacts locally without touching the tracked README image.